# ML-06 — Signal Audit: Do the Flags Hold?

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Distributions

Before testing any belief, look at the fields. Traffic metrics are heavy-tailed, so everything below uses **log scales, medians, or pooled rates** (total clicks ÷ total impressions), never the mean of per-page CTRs.

In [1]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/raw/content_refresh_anonymized.csv").exists())
df = pd.read_csv(root / "data/raw/content_refresh_anonymized.csv")
cols = ["impressions_90d", "clicks_90d", "ctr", "avg_position", "word_count", "content_age_days", "scroll_rate"]
q = df[cols].quantile([0.1, 0.5, 0.9, 0.99]).T
q["mean"], q["max"], q["missing"] = df[cols].mean(), df[cols].max(), df[cols].isna().mean()
print(f"{len(df):,} pages, {df['client_id'].nunique()} clients")
print(q.round(2).to_string())
top1 = df["impressions_90d"].sort_values(ascending=False)
print(f"\nHeavy tail: the top 1% of pages hold {top1.head(len(df)//100).sum()/top1.sum():.0%} of all impressions; "
      f"median clicks in 90 days = {df['clicks_90d'].median():.0f}")
print(f"scroll_rate is exactly 0 for {(df['scroll_rate'] == 0).mean():.0%} of pages and reaches {df['scroll_rate'].max():.0f} "
      f"(a ratio from two systems, so it can exceed 100) -> treat with care")

30,000 pages, 32 clients
                     0.1      0.5       0.9      0.99     mean       max  missing
impressions_90d      5.0   731.00  12136.40  73505.83  5200.37  517715.0     0.00
clicks_90d           0.0     1.00     32.00    253.01    16.10    4178.0     0.00
ctr                  0.0     0.07      0.65      8.33     0.51     100.0     0.00
avg_position         3.7    10.80     36.80     69.90    16.34     245.0     0.00
word_count        1382.0  2877.00   5327.00   7292.00  3107.76    9546.0     0.26
content_age_days   104.0   236.00    463.00    537.00   256.17     564.0     0.00
scroll_rate          0.0     5.00     50.00    100.00    18.21     300.0     0.00

Heavy tail: the top 1% of pages hold 25% of all impressions; median clicks in 90 days = 1
scroll_rate is exactly 0 for 37% of pages and reaches 300 (a ratio from two systems, so it can exceed 100) -> treat with care


## 2. Signal test #1 / #2 / #3 (verdict each)

Slice for all tests: **visible pages**, positions 1–20, ≥ 1,000 impressions in 90 days. Rates are pooled. No verdict from a group under 50 pages.

- **#1 "A better position means a higher CTR."**
- **#2 "Longer content earns more clicks."**
- **#3 "Recently updated pages decline less."**

In [2]:
v = df[(df["avg_position"] > 0) & (df["avg_position"] <= 20) & (df["impressions_90d"] >= 1000)].copy()
pooled = lambda g: pd.Series({"pages": len(g), "pooled_ctr": 100 * g["clicks_90d"].sum() / g["impressions_90d"].sum(),
                              "median_impressions": g["impressions_90d"].median(), "share_declining": (g["trend_direction"] == "down").mean()})
print(f"Visible slice: {len(v):,} pages")

# 1 position -> CTR, narrow buckets, re-run on the two 30-day windows
v["bucket"] = pd.cut(v["avg_position"], [0, 3, 5, 10, 15, 20], labels=["1-3", "3-5", "5-10", "10-15", "15-20"])
t1 = v.groupby("bucket", observed=True).apply(pooled)[["pages", "pooled_ctr"]]
for w in ["prev", "last"]:
    g = v.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    t1[f"ctr_{w}_30d"] = 100 * g.iloc[:, 0] / g.iloc[:, 1]
print("\n#1 Position -> CTR (pooled, %):"); print(t1.round(3).to_string())

# 2 length -> clicks (word count quartiles; missing kept as its own group)
v["length"] = pd.qcut(v["word_count"], 4, labels=["Q1 shortest", "Q2", "Q3", "Q4 longest"]).cat.add_categories("missing").fillna("missing")
t2 = v.groupby("length", observed=True).apply(pooled)
t2["median_clicks"] = v.groupby("length", observed=True)["clicks_90d"].median()
print("\n#2 Word count -> clicks:"); print(t2.round(3).to_string())
print("Spearman(word_count, clicks_90d):", round(v[["word_count", "clicks_90d"]].corr("spearman").iloc[0, 1], 3))

# 3 freshness -> decline
t3 = v.groupby("freshness_tier").apply(pooled)
print("\n#3 Freshness tier -> share of pages declining:"); print(t3.round(3).to_string())
same_age = v[v["content_age_days"].between(91, 365)].groupby("freshness_tier").apply(pooled)
print("\n#3 re-run inside one age band (91-365 days old):"); print(same_age.round(3).to_string())

Visible slice: 10,011 pages

#1 Position -> CTR (pooled, %):
         pages  pooled_ctr  ctr_prev_30d  ctr_last_30d
bucket                                                
1-3      416.0       0.491         0.559         0.570
3-5     1539.0       0.456         0.458         0.480
5-10    4636.0       0.292         0.276         0.301
10-15   2076.0       0.362         0.354         0.410
15-20   1344.0       0.346         0.359         0.429

#2 Word count -> clicks:
              pages  pooled_ctr  median_impressions  share_declining  median_clicks
length                                                                             
Q1 shortest  1729.0       0.325              3450.0            0.630            8.0
Q2           1730.0       0.358              5138.0            0.560           14.0
Q3           1725.0       0.404              5489.0            0.557           15.0
Q4 longest   1727.0       0.470              4880.0            0.647           14.0
missing      3100.0     

**Verdicts**

| # | Claim | Verdict | What it means |
|---|---|---|---|
| 1 | Better position → higher CTR | **MIXED** | True from 1–3 down to 5–10, then it **reverses**: positions 10–20 pooled a *higher* CTR than 5–10, in all three windows. A tier rule that assumes CTR falls with position misjudges pages around 5–20. This is why my lane uses narrow buckets. |
| 2 | Longer content earns more clicks | **CONFIRMED (directional, weak)** | Pooled CTR rises steadily from the shortest to the longest quartile (0.33% → 0.47%); median clicks rise from the shortest quartile and then plateau; the rank correlation is weak (0.11). Association only: longer pages may target different keywords. Pages with no word count (all of them keyword articles) have the lowest CTR, so missingness itself carries a signal. |
| 3 | Recently updated pages decline less | **FALSE (in this slice)** | Among visible pages, 57% of those updated in the last 30 days were "declining" vs 60% for 91–180 days: about the same. Inside one age band (91–365 days) the gap disappears (62% vs 61%). The 31–90 and 181+ tiers have 38 and 6 pages: insufficient data. Updates are chosen by a workflow, so this isn't evidence that updates don't work either, only that this data doesn't show the pattern. |

## 3. The flag-linked test

**The flag:** FlyRank's public report describes an internal "Fix CTR" optimization flag: *visible, but low clicks*. Its working assumption is that such a page **has a real, lasting CTR problem worth an edit**, not a one-month wobble.

**The test:** flag a visible page when its CTR in days 31–60 is below half of its position bucket's pooled CTR (my version of "Fix CTR"). Then check the **next** 30 days: does the flagged page still under-capture, compared with pages that weren't flagged? I also test the coarse-tier version of "expected CTR", since a flag needs a fair baseline.

In [3]:
f = v[(v["impressions_prev_30d"] >= 100) & (v["impressions_last_30d"] >= 100)].copy()
for w in ["prev", "last"]:
    f[f"ctr_{w}"] = 100 * f[f"clicks_{w}_30d"] / f[f"impressions_{w}_30d"]
    g = f.groupby("bucket", observed=True)[[f"clicks_{w}_30d", f"impressions_{w}_30d"]].sum()
    f[f"exp_{w}"] = f["bucket"].map(100 * g.iloc[:, 0] / g.iloc[:, 1]).astype(float)
    f[f"under_{w}"] = (f[f"ctr_{w}"] < 0.5 * f[f"exp_{w}"]).astype(int)
flag = f.groupby(f["under_prev"].map({1: "flagged (Fix CTR-style)", 0: "not flagged"}))["under_last"].agg(still_under_next_month="mean", pages="size")
print("Does the flag hold one month later?"); print(flag.round(3).to_string())
lift = flag.loc["flagged (Fix CTR-style)", "still_under_next_month"] / flag.loc["not flagged", "still_under_next_month"]
print(f"-> flagged pages are {lift:.1f}x as likely to still under-capture; "
      f"{1 - flag.loc['flagged (Fix CTR-style)', 'still_under_next_month']:.0%} of flags recover without an edit")

by_client = f[f["under_prev"] == 1].groupby("client_id")["under_last"].agg(["mean", "size"]).query("size >= 50")
print(f"\nAcross {len(by_client)} clients with >= 50 flagged pages, persistence ranges "
      f"{by_client['mean'].min():.0%}-{by_client['mean'].max():.0%} (median {by_client['mean'].median():.0%})")

tiers = df[df["position_tier"].isin(["top_3", "page_1", "striking"])].groupby("position_tier").apply(
    lambda g: pd.Series({"pages": len(g), "pooled_ctr": 100 * g["clicks_90d"].sum() / g["impressions_90d"].sum()}))
print("\nCoarse-tier 'expected CTR' (dataset tiers, all pages):"); print(tiers.reindex(["top_3", "page_1", "striking"]).round(3).to_string())

Does the flag hold one month later?
                         still_under_next_month  pages
under_prev                                            
flagged (Fix CTR-style)                   0.607   4110
not flagged                               0.209   5554
-> flagged pages are 2.9x as likely to still under-capture; 39% of flags recover without an edit

Across 9 clients with >= 50 flagged pages, persistence ranges 42%-67% (median 57%)

Coarse-tier 'expected CTR' (dataset tiers, all pages):
                 pages  pooled_ctr
position_tier                     
top_3           2321.0       0.488
page_1         11814.0       0.350
striking        7304.0       0.347


**Verdict: CONFIRMED, with a cost.** Flagged pages were about 3× as likely as unflagged ones to still under-capture a month later (61% vs 21%), and persistence stayed between 42% and 67% across the 9 clients with at least 50 flagged pages. So the flag's assumption is supported by this data. But about 4 in 10 flags recovered on their own, and the coarse-tier version of "expected CTR" is shaky: `striking` (positions 11–20) pooled about the same CTR as `page_1` (4–10). A flag built on coarse tiers would over-flag page-one pages and under-flag striking-distance ones.

## 4. What this means in practice

- **Trust the "Fix CTR" idea, but not a single month of it.** A low-CTR-for-position flag points at real, lasting gaps more often than not; about 4 in 10 recover anyway, so a second month of evidence (or a model) before editing saves editor time.
- **Compare a page with its exact neighbourhood, not a coarse tier.** CTR doesn't fall smoothly with position in this data; narrow position buckets give a fairer "expected CTR".
- **Don't sell "refresh = growth" or "longer = better" from this data.** Length shows only a weak association, and recently updated pages didn't decline less at all in this slice.

These verdicts are observed in the 30,000-page starter slice and are decision-support, not causal.

In [4]:
summary = pd.DataFrame([
    ("#1 better position -> higher CTR", "MIXED", f"1-3: {t1.loc['1-3', 'pooled_ctr']:.2f}%  5-10: {t1.loc['5-10', 'pooled_ctr']:.2f}%  10-15: {t1.loc['10-15', 'pooled_ctr']:.2f}%"),
    ("#2 longer content -> more clicks", "CONFIRMED (directional, weak)", f"pooled CTR Q1 {t2.loc['Q1 shortest', 'pooled_ctr']:.2f}% -> Q4 {t2.loc['Q4 longest', 'pooled_ctr']:.2f}%"),
    ("#3 recently updated -> declines less", "FALSE (in this slice)", f"declining: 0-30d {t3.loc['0-30', 'share_declining']:.0%} vs 91-180d {t3.loc['91-180', 'share_declining']:.0%}, same age band: no gap"),
    ("Flag: Fix CTR gap is lasting", "CONFIRMED, with a cost", f"{flag.iloc[0, 0]:.0%} vs {flag.iloc[1, 0]:.0%} still under next month"),
], columns=["signal", "verdict", "evidence (n >= 50 per group)"])
summary

,signal,verdict,evidence (n >= 50 per group)
0,#1 better position -> higher CTR,MIXED,1-3: 0.49% 5-10: 0.29% 10-15: 0.36%
1,#2 longer content -> more clicks,"CONFIRMED (directional, weak)",pooled CTR Q1 0.32% -> Q4 0.47%
2,#3 recently updated -> declines less,FALSE (in this slice),"declining: 0-30d 57% vs 91-180d 60%, same age ..."
3,Flag: Fix CTR gap is lasting,"CONFIRMED, with a cost",61% vs 21% still under next month


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.